# Lab 5: Build a Language Model from Scratch -- microGPT (SOLUTIONS)
## CCI Session 4

**Duration:** 15 minutes  
**No GPU needed** -- this runs on CPU!

### Why This Matters
> Every LLM you've used -- GPT-4o, Claude, Gemini -- is built on the same principles. Today we build one from absolute scratch: no PyTorch, no TensorFlow, no frameworks. Just pure Python. You'll see automatic differentiation, self-attention, and training -- all in ~200 lines. When you understand this, you understand what every AI model is doing.

### Source
Based on [Andrej Karpathy's microgpt.py](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95)

### Objective
- Understand automatic differentiation (backpropagation) from scratch
- See how self-attention works at the lowest level
- Train a tiny language model and watch it generate names
- Connect this to the models you've used throughout the course

---
## Cell 1: Download Data & Setup

We use the **names dataset** from Karpathy's makemore project -- 32,000+ human names.  
Our model will learn to generate new, plausible names character by character.

In [ ]:
# === CELL 1: GET THE DATA ===
import random
import math
import os

random.seed(42)

# Download the names dataset (same one Karpathy uses)
if not os.path.exists('input.txt'):
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')

# SOLUTION: Read the names and explore the data
docs = [line.strip() for line in open('input.txt') if line.strip()]
random.shuffle(docs)
print(f"Number of names: {len(docs)}")
print(f"First 10 names: {docs[:10]}")
print(f"Shortest name: {min(docs, key=len)}")
print(f"Longest name: {max(docs, key=len)}")

---
## Cell 2: Build the Vocabulary

We need to convert characters to numbers (and back) so the model can process them.  
This is exactly what tiktoken does for GPT-4 -- just at character level instead of subword level.

In [ ]:
# === CELL 2: VOCABULARY ===
# Build a character-level vocabulary from the names

# SOLUTION: Create the vocabulary
uchars = sorted(set(''.join(docs)))
BOS = len(uchars)  # Beginning/End of sequence token (like <|endoftext|> in GPT)
vocab_size = len(uchars) + 1
print(f"Unique characters: {uchars}")
print(f"Vocab size: {vocab_size}")
print(f"BOS token index: {BOS}")

# NOTE: GPT-4 uses ~100,000 subword tokens. We use ~27 character tokens.
# Same idea, different granularity.

---
## Cell 3: The Value Class -- Automatic Differentiation from Scratch

This is the **core educational piece**. In PyTorch, you call `loss.backward()` and gradients magically appear. Here we build that magic ourselves.

Every operation (`+`, `*`, `exp`, `log`) records how gradients flow backward through it.  
This IS backpropagation -- the algorithm that trains every neural network.

In [ ]:
# === CELL 3: AUTOMATIC DIFFERENTIATION FROM SCRATCH ===
# In PyTorch, you call loss.backward() and gradients magically appear.
# Here we build that magic ourselves.

class Value:
    """A single scalar value with automatic gradient tracking.
    
    Every Value stores:
    - data: the actual number
    - grad: the gradient (how much the final loss changes if this value changes)
    - _children: the Values that produced this one
    - _local_grads: the local derivatives (chain rule!)
    """
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    # --- Addition: d(a+b)/da = 1, d(a+b)/db = 1 ---
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    # --- Multiplication: d(a*b)/da = b, d(a*b)/db = a ---
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    # --- Power: d(a^n)/da = n * a^(n-1) ---
    def __pow__(self, other):
        return Value(self.data ** other, (self,), (other * self.data ** (other - 1),))

    # --- Logarithm: d(log(a))/da = 1/a ---
    def log(self):
        return Value(math.log(self.data), (self,), (1 / self.data,))

    # --- Exponential: d(exp(a))/da = exp(a) ---
    def exp(self):
        return Value(math.exp(self.data), (self,), (math.exp(self.data),))

    # --- ReLU: d(relu(a))/da = 1 if a > 0, else 0 ---
    def relu(self):
        return Value(max(0, self.data), (self,), (float(self.data > 0),))

    # --- Helper operations ---
    def __neg__(self):          return self * -1
    def __radd__(self, other):  return self + other
    def __sub__(self, other):   return self + (-other)
    def __rsub__(self, other):  return other + (-self)
    def __rmul__(self, other):  return self * other
    def __truediv__(self, other):  return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1

    def backward(self):
        """Backpropagation! Compute gradients for ALL Values in the graph.
        
        1. Topological sort: order nodes so parents come after children
        2. Walk backward: propagate gradients using the chain rule
        """
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1  # dL/dL = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad  # THE CHAIN RULE!

# --- Quick test ---
a = Value(2.0)
b = Value(3.0)
c = a * b + a  # c = 2*3 + 2 = 8
c.backward()
print(f"c = {c.data}")       # Should be 8.0
print(f"dc/da = {a.grad}")   # Should be 4.0 (b + 1 = 3 + 1)
print(f"dc/db = {b.grad}")   # Should be 2.0 (a = 2)

# SOLUTION:
# Q: For a ** 3, what is the local gradient with respect to a?
# A: 3 * a^2 (the power rule from calculus: d/da(a^3) = 3a^2)

---
## Cell 4: Neural Network Building Blocks

Now we build layers using our `Value` class. No PyTorch -- just lists of `Value` objects!

- **Embedding**: converts token IDs to vectors
- **Linear**: matrix multiply (the basic neural network operation)
- **RMSNorm**: stabilizes training
- **Softmax**: converts raw scores to probabilities
- **CausalSelfAttention**: the heart of transformers!

In [ ]:
# === CELL 4: NEURAL NETWORK BUILDING BLOCKS ===
# Layers built using our Value class. No PyTorch!

# Hyperparameters (tiny -- this is a teaching model!)
n_layer = 1       # number of transformer blocks
n_embd = 16       # embedding dimension
block_size = 16   # context window (max sequence length)
n_head = 4        # number of attention heads
head_dim = n_embd // n_head  # dimension per head

# Helper: create a matrix of Value objects (random initialization)
matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]

# Linear layer: y = x @ W  (matrix multiply, but with Value objects)
def linear(x, w):
    """Matrix multiply: each output is a dot product of input with a weight row."""
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

# Softmax: convert raw scores to probabilities
def softmax(logits):
    """Numerically stable softmax."""
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

# RMSNorm: normalize activations (like LayerNorm but simpler)
def rmsnorm(x):
    """Root Mean Square normalization -- stabilizes training."""
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

print("Building blocks defined!")
print(f"  - linear(): matrix multiply with Value objects")
print(f"  - softmax(): convert scores to probabilities")
print(f"  - rmsnorm(): normalize activations")

---
## Cell 5: Initialize All Model Parameters

A transformer has specific weight matrices. We create them all in a dictionary,  
just like PyTorch's `state_dict`. Each weight is a `Value` object that tracks gradients.

In [ ]:
# === CELL 5: MODEL PARAMETERS ===
# Create all the weight matrices for our GPT model.
# This is equivalent to model.state_dict() in PyTorch.

state_dict = {
    'wte': matrix(vocab_size, n_embd),      # Token embedding table
    'wpe': matrix(block_size, n_embd),       # Position embedding table
    'lm_head': matrix(vocab_size, n_embd),   # Output projection (logits)
}

# For each transformer layer, create Q, K, V, output, and MLP weights
for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)   # Query
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)   # Key
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)   # Value
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)   # Output projection
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)  # MLP expand
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)  # MLP contract

# Collect ALL parameters into a flat list (for the optimizer)
params = [p for mat in state_dict.values() for row in mat for p in row]

# SOLUTION: Print the parameter count
print(f"Total parameters: {len(params):,}")
print(f"\nFor comparison:")
print(f"  Our microGPT:  {len(params):,} parameters")
print(f"  GPT-2:         1,500,000,000 parameters")
print(f"  GPT-4:         ~1,800,000,000,000 parameters")
print(f"\nSame architecture, VERY different scale!")

---
## Cell 6: The GPT Forward Pass

This is the complete transformer forward pass:  
**Token Embedding + Position Embedding -> RMSNorm -> Self-Attention -> MLP -> Output**

Read this carefully -- this is what happens every time GPT-4o processes your prompt!

In [ ]:
# === CELL 6: THE GPT FORWARD PASS ===
# This is the complete model.

def gpt(token_id, pos_id, keys, values):
    """
    Process ONE token through the entire GPT model.
    
    Args:
        token_id: which token (character) we're processing
        pos_id: position in the sequence (0, 1, 2, ...)
        keys: cached key vectors from previous tokens (for attention)
        values: cached value vectors from previous tokens (for attention)
    
    Returns:
        logits: raw scores for each possible next token
    """
    # Step 1: Token embedding + Position embedding
    tok_emb = state_dict['wte'][token_id]   # Look up token's vector
    pos_emb = state_dict['wpe'][pos_id]      # Look up position's vector
    x = [t + p for t, p in zip(tok_emb, pos_emb)]  # Combine them
    x = rmsnorm(x)  # Normalize

    # Step 2: Transformer blocks
    for li in range(n_layer):
        # --- Multi-Head Self-Attention ---
        x_residual = x  # Save for residual connection
        x = rmsnorm(x)

        # Compute Query, Key, Value projections
        q = linear(x, state_dict[f'layer{li}.attn_wq'])  # What am I looking for?
        k = linear(x, state_dict[f'layer{li}.attn_wk'])  # What do I contain?
        v = linear(x, state_dict[f'layer{li}.attn_wv'])  # What do I communicate?

        # Cache keys and values (this is the KV-cache you hear about!)
        keys[li].append(k)
        values[li].append(v)

        # Multi-head attention: split into heads, attend, combine
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim  # Head start index
            q_h = q[hs:hs + head_dim]                        # This head's query
            k_h = [ki[hs:hs + head_dim] for ki in keys[li]]  # All cached keys
            v_h = [vi[hs:hs + head_dim] for vi in values[li]]  # All cached values

            # Attention scores: dot product of query with all keys
            attn_logits = [
                sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim ** 0.5
                for t in range(len(k_h))
            ]
            attn_weights = softmax(attn_logits)  # Normalize to probabilities

            # Weighted combination of values
            head_out = [
                sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h)))
                for j in range(head_dim)
            ]
            x_attn.extend(head_out)

        # Project attention output and add residual
        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]  # Residual connection!

        # --- Feed-Forward MLP ---
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])  # Expand to 4x
        x = [xi.relu() for xi in x]                       # Non-linearity
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])  # Contract back
        x = [a + b for a, b in zip(x, x_residual)]       # Residual connection!

    # Step 3: Output logits
    logits = linear(x, state_dict['lm_head'])
    return logits

print("GPT forward pass defined!")
print("This is EXACTLY what happens inside GPT-4o when it processes your prompt.")

# SOLUTIONS:
# Q: Why do we divide by head_dim ** 0.5?
# A: Without scaling, the dot products grow large with embedding dimension,
#    pushing softmax into regions with tiny gradients (near 0 or 1).
#    Dividing by sqrt(head_dim) keeps the variance stable. This is the
#    "Scaled" in "Scaled Dot-Product Attention" from the original paper.
#
# Q: What are Q, K, V intuitively?
# A: Think of it like a search engine:
#    - Query (Q): "What am I looking for?" (the current token's question)
#    - Key (K): "What do I contain?" (each token's label/tag)
#    - Value (V): "What information do I provide?" (the actual content)
#    Q dot K gives relevance scores; those scores weight the V vectors.
#
# Q: Why is KV-caching important?
# A: During generation, we process tokens one at a time. Without caching,
#    we'd need to recompute K and V for ALL previous tokens at every step.
#    With caching, we only compute K,V for the new token and reuse the rest.
#    This makes generation O(n) instead of O(n^2) -- critical for speed.

---
## Cell 7: Generate BEFORE Training

Let's see what the model produces **before** any training.  
It should be complete garbage -- random character sequences!

In [ ]:
# === CELL 7: GENERATE BEFORE TRAINING ===

def generate(num_samples=5, temperature=0.8):
    """Generate names from the model."""
    print(f"Generating {num_samples} names (temperature={temperature}):\n")
    for sample_idx in range(num_samples):
        keys_cache = [[] for _ in range(n_layer)]
        values_cache = [[] for _ in range(n_layer)]
        token_id = BOS  # Start with beginning-of-sequence token
        sample = []
        for pos_id in range(block_size):
            logits = gpt(token_id, pos_id, keys_cache, values_cache)
            probs = softmax([l / temperature for l in logits])
            # Sample from the probability distribution
            token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
            if token_id == BOS:  # End of sequence
                break
            sample.append(uchars[token_id])
        print(f"  {sample_idx + 1}. {''.join(sample)}")

# SOLUTION: Generate 5 names from the UNTRAINED model
print("=== BEFORE TRAINING ===")
generate(5)
print("\nThese should be complete garbage -- random character sequences!")

---
## Cell 8: Training -- Adam Optimizer from Scratch

Even the optimizer is built from scratch -- no `torch.optim`!  
We implement the **Adam** optimizer, the same one used to train GPT-4.

Watch the loss decrease -- the model is **learning to generate names**.

In [ ]:
# === CELL 8: TRAINING ===
# Even the optimizer is from scratch -- no torch.optim!

# Adam optimizer hyperparameters
learning_rate = 0.01
beta1, beta2, eps_adam = 0.85, 0.99, 1e-8

# Adam state: momentum (m) and velocity (v) for each parameter
m = [0.0] * len(params)
v = [0.0] * len(params)

# SOLUTION: Train for 1000 steps
num_steps = 1000

for step in range(num_steps):
    # Pick a training example
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    # Forward pass: process each token, accumulate loss
    keys_cache = [[] for _ in range(n_layer)]
    values_cache = [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys_cache, values_cache)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()  # Cross-entropy loss
        losses.append(loss_t)

    loss = (1 / n) * sum(losses)  # Average loss over sequence

    # Backward pass: compute all gradients (OUR Value class does this!)
    loss.backward()

    # Adam optimizer: update all parameters
    lr_t = learning_rate * (1 - step / num_steps)  # Linear learning rate decay
    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0  # Reset gradient for next step

    if step % 100 == 0 or step == num_steps - 1:
        print(f"Step {step + 1:4d} / {num_steps} | Loss: {loss.data:.4f}")

print("\nTraining complete! The model has learned to generate names.")

---
## Cell 9: Generate AFTER Training

Now let's see what our trained model can produce!  
Compare these with the garbage from Cell 7.

In [ ]:
# === CELL 9: GENERATE AFTER TRAINING ===

# SOLUTION: Generate 10 names from the trained model
print("=== AFTER TRAINING ===")
generate(10)
print("\nThese should look like plausible names now!")
print("Compare with the garbage from Cell 7.")

# Try different temperatures
print("\n=== LOW TEMPERATURE (more conservative) ===")
generate(5, temperature=0.3)

print("\n=== HIGH TEMPERATURE (more creative) ===")
generate(5, temperature=1.5)

---
## Cell 10: Your Turn -- Experiment

Change one hyperparameter and retrain to see its effect.

In [ ]:
# === CELL 10: YOUR TURN -- EXPERIMENT ===

# SOLUTIONS:
#
# 1. How does this tiny model relate to GPT-4o that you used in Session 3?
#    A: It uses the EXACT same architecture -- token embeddings, position
#       embeddings, multi-head self-attention with Q/K/V, feed-forward MLP,
#       residual connections, and RMSNorm. The difference is purely scale:
#       our model has ~4,000 parameters and 16-dim embeddings; GPT-4 has
#       ~1.8 trillion parameters and thousands of dimensions. But the
#       fundamental algorithm is identical.
#
# 2. What would you need to change to train this on clinical text instead of names?
#    A: Three main changes:
#       (a) Replace names.txt with clinical text (e.g., radiology reports)
#       (b) Expand the vocabulary to include all characters in clinical text
#           (or switch to subword tokenization for efficiency)
#       (c) Massively scale up: more parameters, more layers, larger embeddings,
#           longer context window, and much more training data/steps.
#       The architecture itself would stay the same!
#
# 3. Why is understanding this important for a clinical AI developer at KHCC?
#    A: When you understand the internals, you can:
#       - Debug model behavior (why did it hallucinate? attention patterns!)
#       - Choose appropriate models for tasks (small vs. large, context limits)
#       - Understand fine-tuning (which weights are being updated?)
#       - Evaluate AI vendor claims (is their model actually doing what they say?)
#       - Make informed decisions about GPU/compute requirements
#       - Communicate technical concepts to clinical leadership

---
## Clinical Connection

This is the **same architecture** inside every model you've used:
- GPT-4o for clinical extraction (Session 3)
- flan-t5 for summarization (Lesson 3)
- BioBERT for NER (Lesson 2)

The difference is **scale**:

| Model | Parameters | Training Data | Context |
|-------|-----------|--------------|----------|
| **Our microGPT** | ~4,000 | 32K names | 16 characters |
| **GPT-2** | 1.5B | 40 GB (WebText) | 1,024 tokens |
| **GPT-4** | ~1.8T | Terabytes (internet) | 128K tokens |

But attention, embeddings, and the training loop are **identical**.  
Understanding this makes you a more informed clinical AI developer at KHCC.

### What We Built Today (with ZERO dependencies!):
- **Automatic differentiation** (the `Value` class) -- this IS backpropagation
- **Token + position embeddings** -- how models understand input
- **Multi-head self-attention** with Q, K, V -- the heart of transformers
- **Feed-forward MLP** with RMSNorm -- processing after attention
- **Adam optimizer** -- the algorithm that trains every LLM
- **A complete GPT** that generates plausible names!